# Applied Algorithms – Tutorial 5
### Solutions will be discussed on May 28
### Form link: https://forms.gle/gUqpR8t6Sio8VZc28

In [3]:
# simple test() function to check function returns vs. what the function was supposed to return.
def test(got, expected):
    if got == expected:
        prefix = ' OK '
    else:
        prefix = '  X '
    print('%s got: %s expected: %s' % (prefix, repr(got), repr(expected)))



## Assignment 1

(a) Apply the Needleman-Wunsch algorithm to the two strings $v = \mathit{APPLE}$ and $w = \mathit{HAPE}$ to find **all** optimal alignments. The scoring scheme is as follows: $+1$ for match, $-1$ for mismatch and $-1$ for indel.
Specify the following for your solution:
- the full DP matrix,
- all backtracking arrows for the DP matrix,
- all optimal alignments (i.e. both strings written on top of each other in such a way that the characters associated with each other are on top of each other, including the necessary gaps '-').

(b) Implement the Needleman-Wunsch algorithm that takes as input 2 strings $v$ and $w$ and a scoring scheme (consisting of match score, mismatch score, indel score). The function should return 2 values. The first return value should be the optimal score and the second should be a list of two strings that make up an optimal alignment. (e.g. the input `v = 'AC', w = 'TA', match_score = 1, mismatch_score = -1, indel_score = -1` should return `-1, ['-AC', 'TA-']`)

In [26]:
def needleman_wunsch(v, w, match_score, mismatch_score, indel_score, just_one_walk = True):
    n = len(v)
    m = len(w)
    dp = [[0] * (m+1) for _ in range(n+1)]
    dp_walk = [[0] * (m+1) for _ in range(n+1)]

    for i in range(n+1):
        for j in range(m+1):
            if i == 0 and j == 0:
                dp[i][j] = 0
                dp_walk[i][j] = [(0, 0)]
                continue
            if j == 0:
                dp[i][j] = dp[i-1][j] + indel_score
                dp_walk[i][j] = [(-1, 0)]
                continue
            elif i == 0:
                dp[i][j] = dp[i][j-1] + indel_score
                dp_walk[i][j] = [(0, -1)]
                continue
            indel_i = dp[i-1][j] + indel_score
            indel_j = dp[i][j-1] + indel_score
            indel_diag = dp[i-1][j-1] + (match_score if v[i-1] == w[j-1] else mismatch_score)
            indel_max = max(indel_i, indel_j, indel_diag)
            dp[i][j] = indel_max

            # Add all possible walks (multiple can be maximal)
            dp_walk[i][j] = []
            if indel_i == indel_max:
                dp_walk[i][j].append((-1, 0))
            if indel_j == indel_max:
                dp_walk[i][j].append((0, -1))
            if indel_diag == indel_max:
                dp_walk[i][j].append((-1, -1))

    all_strings = []
    def walk_back(i, j, string_v = "", string_w = ""):
        if i == 0 and j == 0:
            all_strings.append((string_v[::-1], string_w[::-1]))
            return
        for walk_i, walk_j in dp_walk[i][j]:
            new_string_v = string_v
            new_string_w = string_w
            if walk_i == 0:
                new_string_v += "-"
                new_string_w += w[j + walk_j]
            elif walk_j == 0:
                new_string_v += v[i + walk_i]
                new_string_w += "-"
            else:
                new_string_v += v[i + walk_i]
                new_string_w += w[j + walk_j]

            walk_back(i + walk_i, j + walk_j, new_string_v, new_string_w)

    walk_back(n, m)

    if just_one_walk:
        one_v, one_w = all_strings[0]
        return dp[n][m], [one_v, one_w]
    return dp[n][m], all_strings


_, all_string = needleman_wunsch("APPLE", "HAPE", 1, -1, -1, False)

for str_v, str_w in all_string:
    print(str_v)
    print(str_w)
    print("")


-APPLE
HAP--E

-APPLE
HA-P-E



In [24]:
test(needleman_wunsch('AAAA', 'AA', 1, -1, 0)[0], 2)
test(needleman_wunsch('AAAA', 'ATA', 2, -9, -1)[0], 1)
test(needleman_wunsch('AGACA', 'ACATA', 1, -2, -9)[0], -1)
test(needleman_wunsch('AGACA', 'ACATA', 1, -2, -9)[1], ['AGACA', 'ACATA'])
test(needleman_wunsch('ATTT', 'TTTA', 1, -2, -2)[1], ['ATTT-', '-TTTA'])

# Only one of the following test cases must be correct, as the instance has several optimal solutions
print(['AAAA', 'AA--'] == needleman_wunsch('AAAA', 'AA', 1, -1, 0)[1])
print(['AAAA', 'A-A-'] == needleman_wunsch('AAAA', 'AA', 1, -1, 0)[1])
print(['AAAA', 'A--A'] == needleman_wunsch('AAAA', 'AA', 1, -1, 0)[1])
print(['AAAA', '-AA-'] == needleman_wunsch('AAAA', 'AA', 1, -1, 0)[1])
print(['AAAA', '-A-A'] == needleman_wunsch('AAAA', 'AA', 1, -1, 0)[1])
print(['AAAA', '--AA'] == needleman_wunsch('AAAA', 'AA', 1, -1, 0)[1])

 OK  got: 2 expected: 2
 OK  got: 1 expected: 1
 OK  got: -1 expected: -1
 OK  got: ['AGACA', 'ACATA'] expected: ['AGACA', 'ACATA']
 OK  got: ['ATTT-', '-TTTA'] expected: ['ATTT-', '-TTTA']
True
False
False
False
False
False


## Assignment 2

Given two strings $v = v_1 v_2 \dots v_{n}$ and $w = w_1 w_2 \dots w_{m}$ and a scoring scheme with match score, mismatch score and indel score. The aim is to calculate a matrix $M$, where the entry $M_{i, j}$ is defined as the score of an optimal global alignment between $v$ and $w$, in which the characters $v_i$ and $w_j$ must be aligned with each other. To determine $M_{i, j}$, the characters $v_i$ and $w_j$ must therefore not be aligned with a gap.

Describe an algorithm that calculates the matrix $M$ in time $\mathcal{O}(n \cdot m)$.

**Hint:** Try to break down the calculation of a matrix entry into smaller steps. First consider how $M$ can be calculated naively (i.e. with a larger running time). To arrive at the efficient total runtime, intermediate results must be reused.

## Assignment 3

(a) We compare global and local alignment for a fixed scoring scheme that assigns $+1$ for matches, $-1$ for mismatches and $-1$ for indels. Let $GA_{1, -1, -1}(v, w)$ and $LA_{1, -1, -1}(v, w)$ be the alignment scores with respect to global (GA) and local (LA) alignment for two strings $v, w$. Which of the following statements are true and which are not? Give a justification or a counterexample.

- $GA_{1, -1, -1}(v, w) \leq LA_{1, -1, -1}(v, w)$ for all strings $v$ and $w$
- $GA_{1, -1, -1}(v, w) \geq LA_{1, -1, -1}(v, w)$ for all strings $v$ and $w$

(b) How does this change if we do not consider a fixed scoring scheme, but match score $\alpha$, mismatch score $\mu$ and indel score $\sigma$ can be arbitrary integers? Give reasons for your answers.